> 📓 **Lesson 1.9 — Part 4 of 4: From Table to Decision**
>
> This notebook was split out of the original single `eda_advanced.ipynb` so each part can be opened and run on its own. If you are starting here rather than at Part 1, run the **Setup** cell below first — it loads the same data used throughout Lesson 1.9.
>
> Other notebooks in this set: `Part_1_time_series.ipynb`, `Part_2_data_integration.ipynb`, `Part_3_aggregation_reporting.ipynb`

# Lesson 1.9: EDA Advanced — Data Wrangling & Analysis

Lesson 1.8 asked *"can I trust this data?"*. This lesson asks the next question:
**what is the pattern, and what should we do about it?**

Clean rows on their own answer nothing. You have to put time on the index, join in the tables that
give the rows meaning, reshape them, and group them. That is the whole job here.

**Structure — the four learning outcomes, in order:**
* **Part 1: Time Series** — *parse* dates, then resample and roll them.
* **Part 2: Data Integration** — *merge* tables, and convert wide ↔ long.
* **Part 3: Aggregation & Reporting** — *aggregate* with `groupby`, `pivot_table`, `crosstab`.
* **Part 4: From Table to Decision** — *apply* all of it to answer the owner's actual question.

**How to read the code cells:** read the `# 👉` comment above each line before you run the cell. The comment says
what the line does in plain English; the output shows you it happened.


> **🧭 Today's flow — 180 minutes.** One business problem, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Time Series | **Parse** dates; `resample`, `rolling`, `shift` | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Integration | **Merge** tables; `melt` / `pivot` (wide ↔ long) | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Aggregation & Reporting | **Aggregate**: `groupby`, `pivot_table`, `crosstab` | 45 min |
> | **Part 4** | From Table to Decision | **Apply** split-apply-combine to the real question | 20 min |
>
> **The spine:** one business problem — *The Daily Grind*, a four-outlet café chain — and one main
> file, `data/daily_sales.csv`, from start to finish. Small hand-built tables appear alongside as
> *drills*: they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`.


### The four beats of every summary

Lesson 1.8 gave you four beats for every fix: **find it → decide → apply → verify.**
Summarising has its own four, and every table we build today follows them:

| Beat | Ask yourself | |
|---|---|---|
| **1. Question** | What decision does this number serve? | *Renew the Marina Bay lease — yes or no?* |
| **2. Grain** | One row per **what**? | *One row per outlet, per month* |
| **3. Aggregation** | Sum, mean or count — and **why that one**? | *Sum for totals, mean for efficiency* |
| **4. Check** | Does the total still tie back? | *Grouped total == ungrouped total* |

Beat 4 is the one everyone skips. In Part 2 it catches a join that silently deletes $61,310.


### Setup

Import the libraries, then load the file we will use all session.


In [ ]:
# 👉 Two toolkits. `pd` and `np` are just short nicknames, so we can type `pd.something`
#    instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np

# 👉 Housekeeping only. Pandas renames a few option strings between versions and shouts
#    about it; this keeps those notices out of our output. Nothing to learn here.
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# 👉 The spine. One row per outlet, per day, per part of the day (Morning/Midday/Evening).
#    18 months of trading for a four-outlet café chain, plus a pop-up kiosk.
#    `parse_dates=["date"]` tells pandas: this column is not text, it is a date. More on that in 1.1.
sales = pd.read_csv("../data/daily_sales.csv", parse_dates=["date"])

sales.head()


In [ ]:
# 👉 The 1.8 habit still applies: look before you leap. Shape, types, holes.
print("rows, columns:", sales.shape)
sales.info()


### 🎬 Why this matters — the flat line that hides everything

**The situation.** *The Daily Grind* runs four cafés in Singapore. Revenue has been flat for two
quarters. The Marina Bay lease is up for renewal this month, rent is $9,600, and the owner has to
sign or walk away. She sends you the sales export and asks one question: **what is going on?**

Run the next two cells. The first is the number she already has. The second is the same number,
split by outlet.

> Do not worry about how these two lines work yet — that is Part 1 and Part 3. Just read the output.


In [ ]:
# 👉 Total revenue per quarter for the whole chain -- the headline the owner already has.
#    (`.to_period("Q")` labels each date with its calendar quarter.)
chain_by_quarter = sales.groupby(sales["date"].dt.to_period("Q"))["revenue_sgd"].sum().round(0)

chain_by_quarter


In [ ]:
# 👉 The same revenue, but one column per outlet. Same data. Same period. Different question.
by_outlet = sales.pivot_table(
    index=sales["date"].dt.to_period("Q"),   # down the side: quarter
    columns="outlet_id",                     # across the top: outlet
    values="revenue_sgd",                    # the number in the middle
    aggfunc="sum",                           # how to squash it: add it up
).round(0)

by_outlet


**Read the second table.** OUT-03 falls from about \$138k a quarter to about \$100k. OUT-04 climbs
from about \$96k to about \$131k. One is dying, one is growing, and they move by almost the same
amount — so the chain total barely twitches.

The flat line was never the story. It was **two opposite stories cancelling out**.

No amount of cleaning would have found this. Cleaning gives you rows you can trust; only grouping
turns them into an answer. Three more things you cannot see yet, and will by the end of the session:

1. OUT-03's fall is not a slope. It is a **step**, on one specific week. (Part 1)
2. There is a fifth outlet in this file that does not exist in the outlet list. (Part 2)
3. OUT-03 is still staffed for the revenue it used to make. (Part 3)

Write those three down. We will tick them off.


---

## Part 4: From Table to Decision

**Learning outcome 4:** *Apply the split-apply-combine pattern to answer complex analytical
questions on real datasets.*

**Goal:** the owner does not want the notebook. She wants one table she can act on. Everything
above was practice for these two cells.

⏱️ ~20 min


In [ ]:
# 👉 Beat 1: the question is "which outlets are getting better or worse, and by how much?"
#    Beat 2: the grain is one row per outlet. Two windows: the two most recent quarters
#    (2025 H1) against the same period a year earlier -- like-for-like, no seasonality excuse.
h1_2025 = sales[(sales["date"] >= "2025-01-01") & (sales["date"] <= "2025-06-30")]
h1_2024 = sales[(sales["date"] >= "2024-01-01") & (sales["date"] <= "2024-06-30")]

summary = pd.DataFrame({
    "rev_2025_h1": h1_2025.groupby("outlet_id")["revenue_sgd"].sum(),
    "rev_2024_h1": h1_2024.groupby("outlet_id")["revenue_sgd"].sum(),
})

# 👉 Beat 3: change is a percentage here, because the outlets are different sizes.
summary["change_pct"] = ((summary["rev_2025_h1"] / summary["rev_2024_h1"] - 1) * 100).round(1)

summary.round(0)


In [ ]:
# 👉 Now attach what it costs to run each one, and what each dollar of rent buys.
decision = (
    summary.merge(outlets.set_index("outlet_id")[["outlet_name", "monthly_rent_sgd", "seats"]],
                  left_index=True, right_index=True, how="left")
    .merge(staffing[["rev_per_staff_hour"]], left_index=True, right_index=True, how="left")
)

# 👉 Six months of rent against six months of revenue: rent as a share of takings.
decision["rent_pct_of_revenue"] = (
    decision["monthly_rent_sgd"] * 6 / decision["rev_2025_h1"] * 100
).round(1)

decision = decision[[
    "outlet_name", "rev_2025_h1", "change_pct", "rent_pct_of_revenue", "rev_per_staff_hour"
]].sort_values("change_pct")

decision.round(1)


**The answer, in three sentences.** Chain revenue is flat because Holland Village's growth is
almost exactly cancelling Marina Bay's decline. Marina Bay's fall is a step dated to the first week
of November 2024, not a drift — and it is now paying the highest rent as a share of takings, while
earning the least per staff hour. The lease decision is therefore not "is the chain healthy?" but
"can Marina Bay's November step be reversed, and if not, is that rent still worth paying?"

Notice what the table does **not** do: it does not say "close Marina Bay". It gives the owner the
three numbers that make her decision, and it makes them comparable.


In [ ]:
# 👉 Save the summary tables for Lesson 1.10, where they become the owner's one slide.
decision.to_csv("../data/lesson19_decision.csv")
monthly_by_outlet.to_csv("../data/lesson19_monthly_by_outlet.csv")

print("saved:")
print("  data/lesson19_decision.csv          <- the decision table")
print("  data/lesson19_monthly_by_outlet.csv <- monthly revenue by outlet")


## 🎯 Wrap-Up

1. **A date is a type.** Nothing about time works until `pd.to_datetime` has run.
2. **`resample` changes the grain of time; `rolling` smooths it.** Use `rolling` to tell a step
   (something happened on a date) from a slope (something is slowly changing).
3. **The `how` of a join is a business decision.** `inner` silently deleted $61,310. Default to
   `left`, then check for nulls with `indicator=True`.
4. **Long format is for computers, wide format is for people.** `melt` before you join; `pivot`
   before you show.
5. **Split-apply-combine is the engine.** `groupby` → `.agg()` → `pivot_table` → `crosstab` are
   four faces of one idea.
6. **Always run beat 4.** Grouped totals must tie back to ungrouped totals, or something was lost.
7. **An aggregate hides as much as it reveals.** A flat chain total was two opposite trends. The
   fix is not more data — it is a finer grain.

**Next Steps:**
- Complete the [Assignment](./assignment.md) — the Q3 review pack.
- Next lesson: **1.10 Data Visualisation & Storytelling** — the owner gets 20 seconds and one
  slide. Your `decision` table has to survive the trip.


### 📦 Appendix — self-study

These are in `reference.md` with worked examples, and are not taught live:

- **Hierarchical (Multi-)indexes:** `set_index` with two columns, `swaplevel`, `sort_index`,
  `.xs()` cross-sections.
- **`concat` vs `merge`:** stacking tables that share a shape, versus joining on a key.
- **`join` vs `merge`:** `join` works on the index by default; `merge` works on columns.
- **Time zones and business days:** `tz_localize`, `tz_convert`, `bdate_range`, custom offsets.
- **`.resample().agg()` and `.rolling().agg()`:** several statistics per bucket or window.
- **`stack` / `unstack`:** the index-level equivalents of `melt` / `pivot`.


---

## 🏁 Lesson 1.9 Complete!

You've covered all four sections:
- **Part 1:** Time Series — parsing dates, `resample`, `rolling`
- **Part 2:** Data Integration — `merge`, `melt`, `pivot`
- **Part 3:** Aggregation & Reporting — `groupby`, `pivot_table`, `crosstab`
- **Part 4:** From Table to Decision — turning the analysis into one decision table

**Next step:** work through `assignment.md`.
Then open **Lesson 1.10 — Data Visualisation & Storytelling**, which turns today's `lesson19_decision.csv` into the owner's one slide.